# Semiconductor Fabrication Yield & Sensor Optimization Analysis

## Project Overview
This notebook analyzes wafer manufacturing defects in a 90nm fabrication process. We aim to identify process parameters (Chamber Pressure, Voltage, Gas Flow) that correlate with yield loss and high defect counts.

### Key Objectives:
1. Data Cleaning & Outlier Removal (IQR Method).
2. Exploratory Data Analysis (EDA) on machine sensors.
3. Correlation analysis between process parameters and wafer thickness.
4. Identifying the root cause of yield degradation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Set professional style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

## 1. Data Loading
We load the data from our SQLite database.

In [ ]:
conn = sqlite3.connect('../wafersense.db')
df = pd.read_sql_query("SELECT * FROM wafers", conn)
df_telemetry = pd.read_sql_query("SELECT * FROM telemetry", conn)
conn.close()

print(f"Loaded {len(df)} wafer records.")
df.head()

## 2. Data Cleaning: Outlier Removal (IQR Method)
We remove extreme anomalies in Gate Oxide Thickness that might skew our yield analysis.

In [ ]:
Q1 = df['Gate_Oxide_Thickness'].quantile(0.25)
Q3 = df['Gate_Oxide_Thickness'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df[(df['Gate_Oxide_Thickness'] >= lower_bound) & (df['Gate_Oxide_Thickness'] <= upper_bound)]
print(f"Original size: {len(df)}, Cleaned size: {len(df_clean)}")

## 3. Exploratory Data Analysis (EDA)
### 3.1 Yield Distribution
Most wafers should have high yield. Let's see the distribution.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_clean['Yield_Percentage'], bins=30, kde=True, color='teal')
plt.title('Distribution of Wafer Yield Percentage')
plt.show()

### 3.2 Correlation Heatmap
Does Chamber Pressure correlate with defects?

In [ ]:
plt.figure(figsize=(10, 8))
cols = ['Yield_Percentage', 'Defect_Count', 'Gate_Oxide_Thickness', 'Chamber_Pressure', 'Voltage', 'Gas_Flow_Rate']
sns.heatmap(df_clean[cols].corr(), annot=True, cmap='RdYlGn', center=0)
plt.title('Process Parameter Correlation Heatmap')
plt.show()

## 4. Root Cause Analysis
Let's visualize the relationship between Chamber Pressure and Gate Oxide Thickness.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='Chamber_Pressure', y='Gate_Oxide_Thickness', hue='Yield_Percentage', alpha=0.5)
plt.title('Effect of Chamber Pressure on Gate Oxide Thickness')
plt.show()